# EXP21D-v9 — Sampler Mechanism and Degree-Stratified Outcome
Post-hoc analysis of corrected EXP 2A predictions and 128-d embeddings. This notebook performs no GNN or XGBoost training.


In [ ]:
%%writefile requirements.txt
numpy>=1.26
pandas>=2.1
scikit-learn>=1.4
matplotlib>=3.8
PyYAML>=6.0


In [ ]:
%%writefile config.kaggle.yaml
experiment:
  id: "EXP21D-v9-SamplerMechanism-3S"
  seed: 42
inputs:
  transactions: "/kaggle/input/credit-card-transactions/credit_card_transactions-ibm_v2.csv"
  exp2a_root: "/kaggle/input/exp21d-v5-sampler-correctness"
  exp2b_root: "/kaggle/input/exp21d-v7-sampler-audit-2b"
time_window: {source_start_year: 2016, reference_date: "2018-01-01", train_days: 300, val_days: 30, gap_days: 8, test_days: 60}
analysis:
  methods: ["uniform", "topology", "importance"]
  degree_axes: ["user", "merchant", "max"]
  low_max: 15
  medium_max: 200
  pca_majority_sample: 25000
  save_embeddings: true
output:
  result_dir: "/kaggle/working/result/exp21d_v9_sampler_mechanism_3s"
  snapshot_source: true


In [ ]:
%%writefile config.py
from pathlib import Path

import yaml


METHODS = ("uniform", "topology", "importance")


def load_config(path: str) -> dict:
    source = Path(path)
    config = yaml.safe_load(source.read_text(encoding="utf-8"))
    analysis = config["analysis"]
    if tuple(analysis["methods"]) != METHODS:
        raise ValueError("Urutan sampler harus uniform, topology, importance")
    if tuple(analysis["degree_axes"]) != ("user", "merchant", "max"):
        raise ValueError("Degree axes harus user, merchant, max")
    if not 0 < int(analysis["low_max"]) < int(analysis["medium_max"]):
        raise ValueError("Batas degree tidak valid")
    return config


In [ ]:
%%writefile artifacts.py
from __future__ import annotations

import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path
from typing import Any


def write_json(path: str | Path, value: Any) -> None:
    target = Path(path)
    target.parent.mkdir(parents=True, exist_ok=True)
    temporary = target.with_suffix(target.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False, default=str) + "\n", encoding="utf-8")
    temporary.replace(target)


def append_event(path: str | Path, event: str, **payload: Any) -> None:
    record = {"timestamp_utc": datetime.now(timezone.utc).isoformat(), "event": event, **payload}
    with Path(path).open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")


def sha256(path: str | Path, block_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while block := handle.read(block_size):
            digest.update(block)
    return digest.hexdigest()


def artifact_index(root: str | Path) -> dict:
    base = Path(root)
    files = [
        {"path": item.relative_to(base).as_posix(), "size_bytes": item.stat().st_size, "sha256": sha256(item)}
        for item in sorted(base.rglob("*"))
        if item.is_file() and item.name != "artifact_index.json"
    ]
    result = {"file_count": len(files), "files": files}
    write_json(base / "artifact_index.json", result)
    return result


In [ ]:
%%writefile alignment.py
from __future__ import annotations

from datetime import timedelta
from pathlib import Path

import numpy as np
import pandas as pd


REQUIRED_COLUMNS = [
    "User", "Merchant Name", "Amount", "Use Chip", "Merchant State", "MCC",
    "Errors?", "Is Fraud?", "Year", "Month", "Day", "Time",
]


def discover_file(hint: str | Path, filename: str) -> Path:
    source = Path(hint)
    if source.is_file() and source.name == filename:
        return source.resolve()
    if source.exists():
        matches = sorted(source.rglob(filename))
        if matches:
            return matches[0].resolve()
    if Path("/kaggle/input").exists():
        matches = sorted(Path("/kaggle/input").rglob(filename))
        if matches:
            return matches[0].resolve()
    raise FileNotFoundError(f"Tidak menemukan {filename} dari {source}")


def load_exp2a_aligned_frames(path: str | Path, window: dict) -> tuple[pd.DataFrame, pd.DataFrame]:
    start = pd.Timestamp(window["reference_date"])
    test_start = start + timedelta(days=int(window["train_days"]) + int(window["val_days"]) + int(window["gap_days"]))
    test_end = test_start + timedelta(days=int(window["test_days"]))
    source_start_year = int(window.get("source_start_year", 2016))
    chunks = []
    for chunk in pd.read_csv(path, usecols=REQUIRED_COLUMNS, chunksize=1_000_000):
        chunk = chunk.loc[chunk["Year"] >= source_start_year].copy()
        if len(chunk):
            chunk["source_row"] = chunk.index.to_numpy(dtype=np.int64)
            chunks.append(chunk)
    if not chunks:
        raise ValueError("Dataset kosong setelah filter tahun")
    frame = pd.concat(chunks, ignore_index=True)
    frame["Time"] = frame["Time"].fillna("00:00")
    frame["datetime"] = pd.to_datetime(
        frame["Year"].astype(str) + "-" + frame["Month"].astype(str).str.zfill(2) + "-"
        + frame["Day"].astype(str).str.zfill(2) + " " + frame["Time"]
    )
    # Harus identik dengan EXP 2A: default quicksort dan hanya key datetime.
    frame = frame.sort_values("datetime").reset_index(drop=True)
    frame["Amount"] = frame["Amount"].astype(str).str.replace("$", "", regex=False).str.replace(",", "", regex=False).astype(float)
    frame["label"] = frame["Is Fraud?"].map({"Yes": 1, "No": 0}).astype(np.int8)
    train = frame.loc[(frame["datetime"] >= start) & (frame["datetime"] < start + timedelta(days=int(window["train_days"])))].copy().reset_index(drop=True)
    test = frame.loc[(frame["datetime"] >= test_start) & (frame["datetime"] < test_end)].copy().reset_index(drop=True)
    return train, test


def reproduce_tabular_features(train: pd.DataFrame, test: pd.DataFrame) -> np.ndarray:
    from sklearn.preprocessing import StandardScaler

    result = test.copy()
    amount_scaler = StandardScaler().fit(train[["Amount"]])
    amount = amount_scaler.transform(test[["Amount"]]).ravel()
    train_log = np.log1p(np.maximum(0.0, train["Amount"].to_numpy())).reshape(-1, 1)
    test_log = np.log1p(np.maximum(0.0, test["Amount"].to_numpy())).reshape(-1, 1)
    log_scaler = StandardScaler().fit(train_log)
    result["Amount"] = amount
    result["amount_log"] = log_scaler.transform(test_log).ravel()
    categorical = ["Use Chip", "Merchant State", "MCC", "Errors?"]
    for column in categorical:
        train_values = train[column].fillna("<missing>").astype(str)
        mapping = {value: index + 1 for index, value in enumerate(sorted(train_values.unique().tolist()))}
        result[column] = test[column].fillna("<missing>").astype(str).map(mapping).fillna(0).astype(int)
    hour = test["datetime"].dt.hour + test["datetime"].dt.minute / 60.0
    dow = test["datetime"].dt.dayofweek
    result["hour_sin"] = np.sin(2 * np.pi * hour / 24.0)
    result["hour_cos"] = np.cos(2 * np.pi * hour / 24.0)
    result["dow_sin"] = np.sin(2 * np.pi * dow / 7.0)
    result["dow_cos"] = np.cos(2 * np.pi * dow / 7.0)
    columns = ["Amount", "amount_log", *categorical, "hour_sin", "hour_cos", "dow_sin", "dow_cos"]
    return result[columns].to_numpy(dtype=np.float32)


def add_degree_columns(test: pd.DataFrame, low_max: int, medium_max: int) -> pd.DataFrame:
    frame = test.copy()
    frame["user_degree"] = frame.groupby("User")["User"].transform("size").astype(np.int32)
    frame["merchant_degree"] = frame.groupby("Merchant Name")["Merchant Name"].transform("size").astype(np.int32)
    frame["max_degree"] = np.maximum(frame["user_degree"], frame["merchant_degree"]).astype(np.int32)
    for axis in ("user", "merchant", "max"):
        values = frame[f"{axis}_degree"]
        frame[f"{axis}_stratum"] = pd.Categorical(
            np.select([values <= low_max, values <= medium_max], ["low", "medium"], default="high"),
            categories=["low", "medium", "high"], ordered=True,
        )
    return frame


In [ ]:
%%writefile metrics.py
from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, confusion_matrix, f1_score, precision_score, recall_score


def safe_auprc(labels: np.ndarray, probabilities: np.ndarray) -> float | None:
    return float(average_precision_score(labels, probabilities)) if np.unique(labels).size == 2 else None


def classification_metrics(labels: np.ndarray, probabilities: np.ndarray, threshold: float) -> dict:
    if len(labels) == 0:
        return {
            "transactions": 0, "fraud": 0, "fraud_rate": None, "auprc": None,
            "f1": None, "precision": None, "recall": None,
            "tn": 0, "fp": 0, "fn": 0, "tp": 0, "threshold": float(threshold),
        }
    predicted = (probabilities >= threshold).astype(np.uint8)
    tn, fp, fn, tp = confusion_matrix(labels, predicted, labels=[0, 1]).ravel()
    return {
        "transactions": int(len(labels)), "fraud": int(labels.sum()), "fraud_rate": float(labels.mean()) if len(labels) else None,
        "auprc": safe_auprc(labels, probabilities),
        "f1": float(f1_score(labels, predicted, zero_division=0)),
        "precision": float(precision_score(labels, predicted, zero_division=0)),
        "recall": float(recall_score(labels, predicted, zero_division=0)),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp), "threshold": float(threshold),
    }


def stratified_metrics(table: pd.DataFrame, methods: tuple[str, ...], axes: tuple[str, ...]) -> pd.DataFrame:
    rows = []
    for axis in axes:
        for stratum in ("low", "medium", "high"):
            subset = table[table[f"{axis}_stratum"] == stratum]
            for method in methods:
                rows.append({
                    "degree_axis": axis, "stratum": stratum, "method": method,
                    **classification_metrics(
                        subset["label"].to_numpy(), subset[f"{method}_probability"].to_numpy(),
                        float(table[f"{method}_threshold"].iloc[0]),
                    ),
                })
    return pd.DataFrame(rows)


def fraud_overlap(table: pd.DataFrame, methods: tuple[str, ...]) -> tuple[pd.DataFrame, pd.DataFrame]:
    fraud = table.loc[table["label"] == 1].copy()
    flags = [fraud[f"{method}_predicted"].to_numpy(dtype=np.uint8) for method in methods]
    pattern = np.array(["".join(map(str, values)) for values in zip(*flags)])
    names = {
        "111": "caught_all", "010": "topology_only", "001": "importance_only", "100": "uniform_only",
        "110": "uniform_topology", "101": "uniform_importance", "011": "topology_importance", "000": "missed_all",
    }
    membership = fraud[["transaction_id", "source_row", "datetime", "user_hash", "merchant_hash", "user_degree", "merchant_degree", "max_degree"]].copy()
    membership["capture_pattern"] = pattern
    membership["capture_category"] = [names[value] for value in pattern]
    summary = membership.groupby(["capture_pattern", "capture_category"], observed=True).size().rename("fraud_cases").reset_index()
    all_patterns = pd.DataFrame({"capture_pattern": list(names), "capture_category": list(names.values())})
    summary = all_patterns.merge(summary, how="left").fillna({"fraud_cases": 0})
    summary["fraud_cases"] = summary["fraud_cases"].astype(int)
    return membership, summary


In [ ]:
%%writefile latent.py
from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA


def latent_diagnostics(embedding: np.ndarray, labels: np.ndarray, method: str) -> dict:
    negative, positive = embedding[labels == 0].astype(np.float64), embedding[labels == 1].astype(np.float64)
    mu0, mu1 = negative.mean(axis=0), positive.mean(axis=0)
    difference = mu1 - mu0
    denominator = negative.var(axis=0) + positive.var(axis=0) + 1e-12
    fisher = difference**2 / denominator
    return {
        "method": method, "dimensions": int(embedding.shape[1]),
        "negative_count": int(len(negative)), "positive_count": int(len(positive)),
        "centroid_euclidean_distance": float(np.linalg.norm(difference)),
        "fisher_sum": float(fisher.sum()), "fisher_mean": float(fisher.mean()),
        "fisher_median": float(np.median(fisher)), "fisher_max": float(fisher.max()),
        "top_fisher_dimensions": np.argsort(fisher)[-10:][::-1].astype(int).tolist(),
    }


def pca_projection(
    embedding: np.ndarray, labels: np.ndarray, transaction_ids: np.ndarray,
    method: str, majority_sample: int, seed: int,
) -> tuple[pd.DataFrame, dict]:
    rng = np.random.default_rng(seed)
    positive = np.flatnonzero(labels == 1)
    negative_all = np.flatnonzero(labels == 0)
    negative = np.sort(rng.choice(negative_all, size=min(majority_sample, len(negative_all)), replace=False))
    selected = np.sort(np.concatenate([positive, negative]))
    model = PCA(n_components=2, svd_solver="randomized", random_state=seed)
    coordinates = model.fit_transform(embedding[selected])
    frame = pd.DataFrame({
        "method": method, "transaction_id": transaction_ids[selected], "label": labels[selected],
        "pca_1": coordinates[:, 0], "pca_2": coordinates[:, 1],
    })
    metadata = {
        "method": method, "sample_size": int(len(selected)), "fraud": int(labels[selected].sum()),
        "explained_variance_ratio": model.explained_variance_ratio_.astype(float).tolist(),
        "interpretation": "exploratory_only_separate_pca_basis_per_sampler",
    }
    return frame, metadata


In [ ]:
%%writefile visualize_results.py
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


COLORS = {"uniform": "#4472C4", "topology": "#ED7D31", "importance": "#70AD47"}


def save(fig, output: Path, stem: str) -> list[str]:
    names = []
    for suffix in ("png", "pdf"):
        target = output / f"{stem}.{suffix}"
        fig.savefig(target, dpi=300, bbox_inches="tight")
        names.append(target.name)
    plt.close(fig)
    return names


def create_visualizations(result_dir: str | Path) -> None:
    root = Path(result_dir)
    output = root / "figures"
    output.mkdir(exist_ok=True)
    metrics = pd.read_csv(root / "degree_stratified_metrics.csv")
    overlap = pd.read_csv(root / "fraud_overlap_summary.csv")
    latent = pd.read_csv(root / "latent_diagnostics.csv")
    pca = pd.read_csv(root / "pca_projection.csv.gz")
    generated = []

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
    for axis_plot, degree_axis in zip(axes, ("user", "merchant", "max")):
        subset = metrics[metrics["degree_axis"] == degree_axis]
        x = np.arange(3)
        for offset, method in zip((-0.25, 0.0, 0.25), COLORS):
            values = subset[subset["method"] == method].set_index("stratum").reindex(["low", "medium", "high"])["auprc"]
            axis_plot.bar(x + offset, values, width=0.24, label=method.title(), color=COLORS[method])
        axis_plot.set_xticks(x, ["Low", "Medium", "High"])
        axis_plot.set_title(f"{degree_axis.title()} degree")
        axis_plot.set_ylim(0, 1)
        axis_plot.grid(axis="y", alpha=0.25)
    axes[0].set_ylabel("AUPRC")
    handles, labels = axes[-1].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=3, frameon=False, bbox_to_anchor=(0.5, 0.98))
    fig.suptitle("EXP 3S — Sampler performance by frozen degree strata")
    fig.subplots_adjust(top=0.80)
    generated += save(fig, output, "fig1_degree_stratified_auprc")

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
    for axis_plot, method in zip(axes, COLORS):
        subset = metrics[(metrics["degree_axis"] == "max") & (metrics["method"] == method)].set_index("stratum").reindex(["low", "medium", "high"])
        x = np.arange(3)
        axis_plot.bar(x - 0.18, subset["fp"], width=0.36, label="FP", color="#C55A11")
        axis_plot.bar(x + 0.18, subset["fn"], width=0.36, label="FN", color="#A5A5A5")
        axis_plot.set_xticks(x, ["Low", "Medium", "High"])
        axis_plot.set_title(method.title())
        axis_plot.grid(axis="y", alpha=0.25)
    axes[0].set_ylabel("Error count")
    axes[-1].legend(frameon=False)
    fig.suptitle("False positives and false negatives by max-degree stratum")
    generated += save(fig, output, "fig2_degree_stratified_errors")

    ordered = ["caught_all", "topology_only", "importance_only", "uniform_only", "uniform_topology", "uniform_importance", "topology_importance", "missed_all"]
    values = overlap.set_index("capture_category").reindex(ordered)["fraud_cases"].fillna(0)
    fig, axis_plot = plt.subplots(figsize=(10, 4.8))
    axis_plot.bar(np.arange(len(values)), values, color="#5B9BD5")
    axis_plot.set_xticks(np.arange(len(values)), [name.replace("_", " ").title() for name in values.index], rotation=25, ha="right")
    axis_plot.set_ylabel("Fraud cases")
    axis_plot.set_title("Paired fraud capture overlap at validation-selected thresholds")
    axis_plot.grid(axis="y", alpha=0.25)
    generated += save(fig, output, "fig3_fraud_capture_overlap")

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    for axis_plot, method in zip(axes, COLORS):
        subset = pca[pca["method"] == method]
        normal = subset[subset["label"] == 0]
        fraud = subset[subset["label"] == 1]
        axis_plot.scatter(normal["pca_1"], normal["pca_2"], s=3, alpha=0.08, color="#7F7F7F", rasterized=True)
        axis_plot.scatter(fraud["pca_1"], fraud["pca_2"], s=10, alpha=0.65, color="#C00000", label="Fraud", rasterized=True)
        axis_plot.set_title(method.title())
        axis_plot.set_xlabel("PC1")
        axis_plot.grid(alpha=0.15)
    axes[0].set_ylabel("PC2")
    axes[-1].legend(frameon=False)
    fig.suptitle("Exploratory PCA of 128-d GraphSAGE embeddings (separate basis per sampler)")
    generated += save(fig, output, "fig4_embedding_pca_exploratory")

    metrics.to_csv(output / "data_degree_stratified_metrics.csv", index=False)
    overlap.to_csv(output / "data_fraud_overlap_summary.csv", index=False)
    latent.to_csv(output / "data_latent_diagnostics.csv", index=False)
    pca.to_csv(output / "data_pca_projection.csv.gz", index=False, compression="gzip")
    (output / "figures_manifest.json").write_text(json.dumps({"figures": generated, "source_data": sorted(p.name for p in output.glob("data_*"))}, indent=2) + "\n", encoding="utf-8")


In [ ]:
%%writefile main.py
from __future__ import annotations

import argparse
import hashlib
import importlib.metadata
import json
import platform
import shutil
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

from alignment import add_degree_columns, discover_file, load_exp2a_aligned_frames, reproduce_tabular_features
from artifacts import append_event, artifact_index, sha256, write_json
from config import METHODS, load_config
from latent import latent_diagnostics, pca_projection
from metrics import fraud_overlap, stratified_metrics
from visualize_results import create_visualizations


def resolve_path(value: str, config_path: Path) -> Path:
    path = Path(value)
    return path if path.is_absolute() else (config_path.parent / path).resolve()


def git_revision() -> str | None:
    try:
        return subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
    except (OSError, subprocess.CalledProcessError):
        return None


def entity_hash(values: pd.Series) -> pd.Series:
    unique = values.astype(str).unique()
    mapping = {value: hashlib.sha256(f"EXP21D-3S|{value}".encode()).hexdigest()[:16] for value in unique}
    return values.astype(str).map(mapping)


def locate_inputs(config: dict, config_path: Path) -> dict[str, Path]:
    hints = config["inputs"]
    dataset = discover_file(resolve_path(hints["transactions"], config_path), "credit_card_transactions-ibm_v2.csv")
    exp2a_hint = resolve_path(hints["exp2a_root"], config_path)
    uniform_prediction = discover_file(exp2a_hint, "exp21d_v5_uniform_predictions.npz")
    exp2a_root = uniform_prediction.parent
    exp2b_hint = resolve_path(hints["exp2b_root"], config_path)
    root_cohorts = discover_file(exp2b_hint, "root_cohorts.npz")
    return {"dataset": dataset, "exp2a_root": exp2a_root, "exp2b_root_cohorts": root_cohorts}


def load_method_artifacts(root: Path, method: str) -> dict:
    prediction_path = root / f"exp21d_v5_{method}_predictions.npz"
    feature_path = root / "features" / method / "test_hybrid_features.npy"
    contract_path = root / "features" / method / "feature_contract.json"
    for path in (prediction_path, feature_path, contract_path):
        if not path.is_file():
            raise FileNotFoundError(path)
    with np.load(prediction_path) as values:
        probabilities = values["hybrid_test_probabilities"].astype(np.float32)
        labels = values["test_labels"].astype(np.uint8)
        threshold = float(values["hybrid_optimal_threshold"])
    contract = json.loads(contract_path.read_text(encoding="utf-8"))
    features = np.load(feature_path, mmap_mode="r")
    if features.shape[1] != int(contract["hybrid_dimensions"]):
        raise ValueError(f"Feature contract mismatch untuk {method}")
    return {
        "prediction_path": prediction_path, "feature_path": feature_path, "contract_path": contract_path,
        "probabilities": probabilities, "labels": labels, "threshold": threshold,
        "features": features, "split_point": int(contract["split_point"]),
    }


def write_report(run_dir: Path, table: pd.DataFrame, metrics: pd.DataFrame, overlap: pd.DataFrame, latent: pd.DataFrame) -> None:
    lines = [
        "# EXP21D-v9 — Sampler Mechanism and Degree-Stratified Outcome", "",
        f"Test rows: {len(table):,}; fraud: {int(table['label'].sum()):,}.", "",
        "Degree strata are frozen as low <= 15, medium 16-200, and high > 200.", "",
        "## Fraud capture overlap", "",
    ]
    for row in overlap.itertuples(index=False):
        lines.append(f"- {row.capture_category}: {row.fraud_cases}")
    lines += ["", "## Latent diagnostics", ""]
    for row in latent.itertuples(index=False):
        lines.append(f"- {row.method}: centroid distance {row.centroid_euclidean_distance:.6f}; Fisher sum {row.fisher_sum:.6f}.")
    lines += ["", "All row-level probabilities, predictions, strata, memberships, metrics, PCA source data, and immutable input fingerprints are retained.", ""]
    (run_dir / "analysis_report.md").write_text("\n".join(lines), encoding="utf-8")


def run(config: dict, config_path: Path, output_root: Path) -> Path:
    seed = int(config["experiment"]["seed"])
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    run_dir = output_root / "runs" / f"exp21d_v9_3s_seed{seed}_{stamp}"
    run_dir.mkdir(parents=True, exist_ok=False)
    events = run_dir / "events.jsonl"
    manifest = {
        "experiment_id": config["experiment"]["id"], "status": "running", "seed": seed,
        "scientific_status": "paper_candidate_single_seed_mechanism_analysis",
        "created_at_utc": datetime.now(timezone.utc).isoformat(), "git_revision": git_revision(),
        "environment": {"python": sys.version, "platform": platform.platform(), **{
            name: importlib.metadata.version(name) for name in ("numpy", "pandas", "scikit-learn", "matplotlib", "PyYAML")
        }},
    }
    write_json(run_dir / "manifest.json", manifest)
    (run_dir / "resolved_config.yaml").write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
    append_event(events, "run_started")
    try:
        inputs = locate_inputs(config, config_path)
        append_event(events, "inputs_resolved", **{key: str(value) for key, value in inputs.items()})
        methods = {method: load_method_artifacts(inputs["exp2a_root"], method) for method in METHODS}
        train, test = load_exp2a_aligned_frames(inputs["dataset"], config["time_window"])
        labels = test["label"].to_numpy(dtype=np.uint8)
        if not all(np.array_equal(labels, item["labels"]) for item in methods.values()):
            raise ValueError("Rekonstruksi row alignment tidak cocok dengan label EXP 2A")
        expected_tabular = reproduce_tabular_features(train, test)
        alignment_errors = {}
        for method, item in methods.items():
            observed = np.asarray(item["features"][:, : item["split_point"]])
            error = float(np.max(np.abs(observed - expected_tabular)))
            alignment_errors[method] = error
            if not np.allclose(observed, expected_tabular, rtol=1e-5, atol=1e-5):
                raise ValueError(f"Tabular row alignment gagal untuk {method}; max abs error={error}")
        append_event(events, "row_alignment_validated", rows=len(test), max_abs_errors=alignment_errors)

        analysis = config["analysis"]
        table = add_degree_columns(test, int(analysis["low_max"]), int(analysis["medium_max"]))
        table["transaction_id"] = np.arange(len(table), dtype=np.int32)
        table["user_hash"] = entity_hash(table["User"])
        table["merchant_hash"] = entity_hash(table["Merchant Name"])
        keep = [
            "transaction_id", "source_row", "datetime", "user_hash", "merchant_hash", "label",
            "user_degree", "merchant_degree", "max_degree", "user_stratum", "merchant_stratum", "max_stratum",
        ]
        table = table[keep].copy()
        for method, item in methods.items():
            table[f"{method}_probability"] = item["probabilities"]
            table[f"{method}_threshold"] = item["threshold"]
            table[f"{method}_predicted"] = (item["probabilities"] >= item["threshold"]).astype(np.uint8)

        degree_metrics = stratified_metrics(table, METHODS, tuple(analysis["degree_axes"]))
        membership, overlap = fraud_overlap(table, METHODS)
        latent_rows, pca_frames, pca_meta = [], [], []
        for index, (method, item) in enumerate(methods.items()):
            embedding = np.asarray(item["features"][:, item["split_point"] :], dtype=np.float32)
            latent_rows.append(latent_diagnostics(embedding, labels, method))
            projection, metadata = pca_projection(
                embedding, labels, table["transaction_id"].to_numpy(), method,
                int(analysis["pca_majority_sample"]), seed + index,
            )
            pca_frames.append(projection)
            pca_meta.append(metadata)
        latent_frame = pd.DataFrame(latent_rows)
        pca_frame = pd.concat(pca_frames, ignore_index=True)

        table.to_csv(run_dir / "row_level_analysis.csv.gz", index=False, compression="gzip")
        np.savez_compressed(
            run_dir / "row_level_analysis.npz", transaction_id=table["transaction_id"], source_row=table["source_row"],
            timestamp_ns=table["datetime"].astype("int64"), label=labels,
            user_degree=table["user_degree"], merchant_degree=table["merchant_degree"], max_degree=table["max_degree"],
            **{f"{method}_probability": methods[method]["probabilities"] for method in METHODS},
            **{f"{method}_predicted": table[f"{method}_predicted"].to_numpy(dtype=np.uint8) for method in METHODS},
        )
        degree_metrics.to_csv(run_dir / "degree_stratified_metrics.csv", index=False)
        membership.to_csv(run_dir / "fraud_overlap_membership.csv", index=False)
        np.savez_compressed(
            run_dir / "fraud_overlap_membership.npz", transaction_id=membership["transaction_id"].to_numpy(),
            source_row=membership["source_row"].to_numpy(), capture_pattern=membership["capture_pattern"].to_numpy(dtype="U3"),
            user_degree=membership["user_degree"].to_numpy(), merchant_degree=membership["merchant_degree"].to_numpy(),
        )
        overlap.to_csv(run_dir / "fraud_overlap_summary.csv", index=False)
        latent_frame.to_csv(run_dir / "latent_diagnostics.csv", index=False)
        pca_frame.to_csv(run_dir / "pca_projection.csv.gz", index=False, compression="gzip")
        write_json(run_dir / "pca_metadata.json", pca_meta)
        input_manifest = {
            "dataset": {"path": str(inputs["dataset"]), "size_bytes": inputs["dataset"].stat().st_size},
            "exp2b_root_cohorts": {"path": str(inputs["exp2b_root_cohorts"]), "sha256": sha256(inputs["exp2b_root_cohorts"])},
            "methods": {method: {
                "predictions": {"path": str(item["prediction_path"]), "sha256": sha256(item["prediction_path"])},
                "features": {"path": str(item["feature_path"]), "sha256": sha256(item["feature_path"]), "shape": list(item["features"].shape)},
                "threshold": item["threshold"], "split_point": item["split_point"],
            } for method, item in methods.items()},
            "alignment_gate": {"labels_exact": True, "tabular_max_abs_error": alignment_errors},
        }
        write_json(run_dir / "input_manifest.json", input_manifest)
        write_report(run_dir, table, degree_metrics, overlap, latent_frame)
        create_visualizations(run_dir)
        if config["output"].get("snapshot_source", True):
            snapshot = run_dir / "code_snapshot"
            snapshot.mkdir()
            for item in Path(__file__).resolve().parent.iterdir():
                if item.is_file() and item.suffix.lower() in {".py", ".yaml", ".md", ".txt"}:
                    shutil.copy2(item, snapshot / item.name)
        manifest.update({
            "status": "complete", "completed_at_utc": datetime.now(timezone.utc).isoformat(),
            "test_rows": int(len(table)), "fraud": int(labels.sum()), "alignment_gate_passed": True,
        })
        write_json(run_dir / "manifest.json", manifest)
        append_event(events, "run_complete", rows=len(table), fraud=int(labels.sum()))
        artifact_index(run_dir)
        return run_dir
    except Exception as error:
        manifest.update({"status": "failed", "failed_at_utc": datetime.now(timezone.utc).isoformat(), "error": repr(error)})
        write_json(run_dir / "manifest.json", manifest)
        append_event(events, "run_failed", error=repr(error))
        artifact_index(run_dir)
        raise


def main() -> None:
    parser = argparse.ArgumentParser(description="EXP 3S sampler mechanism and degree-stratified outcome")
    parser.add_argument("--config", default="config.reference.yaml")
    args = parser.parse_args()
    config_path = Path(args.config).resolve()
    config = load_config(str(config_path))
    output = resolve_path(config["output"]["result_dir"], config_path)
    started = time.time()
    run_dir = run(config, config_path, output)
    print(run_dir)
    print(f"runtime_seconds={time.time() - started:.3f}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile validate_mechanism.py
from __future__ import annotations

import inspect
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd

from alignment import add_degree_columns
from artifacts import write_json
from latent import latent_diagnostics, pca_projection
from metrics import fraud_overlap, stratified_metrics
from visualize_results import create_visualizations


def main() -> None:
    rng = np.random.default_rng(42)
    size = 1200
    labels = np.zeros(size, dtype=np.uint8)
    labels[rng.choice(size, 60, replace=False)] = 1
    table = pd.DataFrame({
        "User": np.repeat(np.arange(60), 20), "Merchant Name": np.tile(np.arange(100), 12),
        "label": labels, "transaction_id": np.arange(size, dtype=np.int32),
        "source_row": np.arange(size, dtype=np.int64), "datetime": pd.date_range("2018-12-05", periods=size, freq="min"),
        "user_hash": [f"u{i % 60}" for i in range(size)], "merchant_hash": [f"m{i % 100}" for i in range(size)],
    })
    table = add_degree_columns(table, 15, 200)
    # Force all frozen strata into the validation table.
    table.loc[:99, "user_degree"] = 10
    table.loc[100:499, "user_degree"] = 100
    table.loc[500:, "user_degree"] = 300
    table["user_stratum"] = pd.cut(table["user_degree"], [0, 15, 200, np.inf], labels=["low", "medium", "high"])
    table["max_degree"] = np.maximum(table["user_degree"], table["merchant_degree"])
    table["max_stratum"] = pd.cut(table["max_degree"], [0, 15, 200, np.inf], labels=["low", "medium", "high"])
    for index, method in enumerate(("uniform", "topology", "importance")):
        score = np.clip(0.02 + 0.85 * labels + rng.normal(0, 0.12 + index * 0.01, size), 0, 1)
        table[f"{method}_probability"] = score
        table[f"{method}_threshold"] = 0.5
        table[f"{method}_predicted"] = (score >= 0.5).astype(np.uint8)
    metrics = stratified_metrics(table, ("uniform", "topology", "importance"), ("user", "merchant", "max"))
    membership, overlap = fraud_overlap(table, ("uniform", "topology", "importance"))
    embedding = rng.normal(size=(size, 128)).astype(np.float32) + labels[:, None] * 0.15
    diagnostic = latent_diagnostics(embedding, labels, "uniform")
    projection, metadata = pca_projection(embedding, labels, table["transaction_id"].to_numpy(), "uniform", 200, 42)
    with tempfile.TemporaryDirectory() as temporary:
        root = Path(temporary)
        metrics.to_csv(root / "degree_stratified_metrics.csv", index=False)
        overlap.to_csv(root / "fraud_overlap_summary.csv", index=False)
        pd.DataFrame([diagnostic] * 3).assign(method=["uniform", "topology", "importance"]).to_csv(root / "latent_diagnostics.csv", index=False)
        pd.concat([projection.assign(method=method) for method in ("uniform", "topology", "importance")]).to_csv(root / "pca_projection.csv.gz", index=False, compression="gzip")
        create_visualizations(root)
        figures_valid = len(list((root / "figures").glob("*.png"))) == 4 and len(list((root / "figures").glob("*.pdf"))) == 4
    source = inspect.getsource(fraud_overlap)
    checks = {
        "strata_contract": set(metrics["stratum"]) == {"low", "medium", "high"},
        "all_axes_present": set(metrics["degree_axis"]) == {"user", "merchant", "max"},
        "all_methods_present": set(metrics["method"]) == {"uniform", "topology", "importance"},
        "fraud_membership_complete": len(membership) == int(labels.sum()),
        "eight_capture_patterns_reported": len(overlap) == 8,
        "threshold_predictions_used": "_predicted" in source,
        "latent_dimensions_correct": diagnostic["dimensions"] == 128,
        "fisher_is_finite": np.isfinite(diagnostic["fisher_sum"]),
        "pca_contains_all_fraud": int(projection["label"].sum()) == int(labels.sum()),
        "pca_marked_exploratory": metadata["interpretation"].startswith("exploratory_only"),
        "all_visualizations_created": figures_valid,
    }
    for name, passed in checks.items():
        print(f"[{'PASS' if passed else 'FAIL'}] {name}")
    write_json(Path(__file__).resolve().parent / "validation_report.json", {"status": "passed" if all(checks.values()) else "failed", "checks": {key: bool(value) for key, value in checks.items()}})
    if not all(checks.values()):
        raise SystemExit(1)


if __name__ == "__main__":
    main()


In [ ]:
import importlib.util
import subprocess
import sys
required={'numpy':'numpy','pandas':'pandas','sklearn':'scikit-learn','matplotlib':'matplotlib','yaml':'PyYAML'}
missing=[package for module,package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable,'-m','pip','install','-q',*missing],check=True)


In [ ]:
!python validate_mechanism.py


In [ ]:
!python main.py --config config.kaggle.yaml
